# AutoGSQ-RCO — Qwen3-4B full pipeline on Colab

End-to-end: `generate-db` (36 layers) → `allocate` → `assemble` → PPL eval.
Forked from the local Windows pipeline so the RTX A4000 stays free.

**Honest expectations before you press Run all:**

- The full 36-layer DB build takes **~6+ hours** (local A4000 pace is ~10 min/layer; a Colab T4 is slower). Free-tier Colab sessions cap at ~12h and can preempt idle — this notebook survives that: the DB lives on **Google Drive** and `generate-db` resumes (`progress.json`), so you just re-run the build cell after a reconnect.
- Colab Pro / background execution recommended for a single-session run. Free tier works, likely across 2+ sessions.
- Qwen3-4B in bf16 is ~8 GB; fits a T4 (16 GB). Calibration + per-layer training are streamed one layer at a time, same as locally.

In [ ]:
# 0 — GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
# 1 — Mount Drive: the candidate DB (multiple GB) must survive session preempts.
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/autogsq_rco'
DB = f'{ROOT}/qwen3_4b_db'
!mkdir -p "$ROOT"

In [ ]:
# 2 — Clone + install. Push this repo to GitHub first, then store its URL
# in Colab Secrets as REPO_URL (never paste URLs/tokens into the notebook).
# NOTE: `datasets` is required for wikitext2 calibration but is not
# declared in pyproject.toml, so it is pinned explicitly here.
try:
    from google.colab import userdata
    _url = userdata.get('REPO_URL')
except Exception:
    _url = None
if not _url:
    raise SystemExit('Set REPO_URL in Colab Secrets, then re-run this cell.')
%cd /content
!test -d AutoGSQ-RCO || git clone {_url} AutoGSQ-RCO
%cd /content/AutoGSQ-RCO
!pip install -q -e . "datasets>=3.0.0"

In [ ]:
# 3 — Optional HF login (raises quotas; harmless if skipped). Token stored via
# Colab Secrets as HF_TOKEN, never pasted into the notebook.
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'))
    print('HF login OK')
except Exception as e:
    print(f'skipping HF login ({e}); continuing unauthenticated')

In [ ]:
# 4 — Colab config. Same hyperparams as the local probe (runtime/qwen3_4b_probe.yaml),
# except model.name is the HF hub ID (loader takes hub IDs directly) and all
# outputs point at Drive. `device: cuda` = the Colab GPU.
config = '''model:
  name: "Qwen/Qwen3-4B"
  device: "cuda"
  dtype: "bfloat16"

data:
  dataset_name: "wikitext2"
  num_samples: 64
  max_length: 1024

gsq:
  bitwidth_options: [2, 3, 4, "ternary"]
  init_method: "gptq"
  group_size: 128
  temperature: [2.0, 0.05]
  logit_scale: [100.0, 500.0]
  num_epochs: 4
  optimizer: "lion"
  learning_rate: 0.001
  steps_per_epoch: 32
  lr_logits: 0.0002
  lr_scales: 0.0001
  gs_weight_decay: 1.0
  warmup_ratio: 0.1
  logits_dtype: "bfloat16"
  gptq_nsamples: 64
  gptq_damping: 0.01
  calib_max_length: 1024
  calib_device: "cuda"

rco:
  target_bpw: 2.75
  eval_dataset: "wikitext2"
  num_steps: 300
  lr: 0.01
  temperature: 1.0
  temperature_min: 0.1
  temperature_schedule: "linear"
  refine_steps: 20
  refine_batches: 2
  refine_max_length: 256

gguf:
  gguf_type_tolerance_bpw: 0.15
  fallback_on_no_match: "round_down"

output:
  checkpoint_dir: "/content/drive/MyDrive/autogsq_rco/qwen3_4b_colab"
  run_id: null
'''
open('/content/qwen3_4b_colab.yaml', 'w').write(config)
print('wrote /content/qwen3_4b_colab.yaml')

In [ ]:
# 5 — Stage 1: candidate DB build. THE long cell (hours). Resume is on by
# default: if Colab kills the session, reconnect + re-run from cell 1,
# then re-run THIS cell; finished layers print
# "already complete, skipping" and it continues where it stopped.
# Smoke-test first with `--max-layers 2` (~20 min) before the full run.
!autogsq generate-db --config /content/qwen3_4b_colab.yaml --out-dir "$ROOT/qwen3_4b_db"

In [ ]:
# 6 — Progress check (cheap, re-run anytime)
!autogsq status --db-dir "$ROOT/qwen3_4b_db"

In [ ]:
# 7 — Stage 2: RCO allocation at 2.75 bpw (minutes, CPU)
!autogsq allocate --db-dir "$ROOT/qwen3_4b_db" --config /content/qwen3_4b_colab.yaml --target-bpw 2.75 --out "$ROOT/qwen3_4b_alloc_full.json"

In [ ]:
# 8 — Stage 3: assemble the GGUF. Colab VMs have 2 vCPUs, so -j 2.
# (~15-20 min on 16 local cores; expect longer here.)
!autogsq assemble --db-dir "$ROOT/qwen3_4b_db" --allocation "$ROOT/qwen3_4b_alloc_full.json" --output "$ROOT/qwen3_4b_colab/model-full.gguf" --config /content/qwen3_4b_colab.yaml -j 2

In [ ]:
# 9 — Stage 4: PPL eval. --source takes the hub ID directly (lazy weight
# loading, no local copy needed). Local baseline for comparison: 14.21
# (official Q4_K_M). Our local RCO number at probe scale: 42.67.
!python src/autogsq/eval/gguf_ppl.py --gguf "$ROOT/qwen3_4b_colab/model-full.gguf" --source Qwen/Qwen3-4B --max-windows 20

In [ ]:
# 10 — Artifacts
!ls -lh "$ROOT/qwen3_4b_colab/" "$ROOT/qwen3_4b_alloc_full.json"